## C. Generative Models

### 3.9 GAN

> **做什么**：生成器造假图，判别器辨真假，对抗训练提升质量  
> **经典案例**：DCGAN生成MNIST手写数字

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cpu')
nz = 64  # 噪声向量维度

# 数据加载
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5],[0.5])])
dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 生成器G：噪声->假图像
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nz, 256), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Linear(512, 28*28), nn.Tanh()  # 输出范围[-1,1]
        )
    def forward(self, z): return self.net(z).view(-1,1,28,28)

# 判别器D：图像->真假概率
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, img): return self.net(img.view(-1, 28*28))

G, D = Generator().to(device), Discriminator().to(device)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

for epoch in range(10):
    for real_imgs, _ in loader:
        real = real_imgs.to(device)
        bs = real.size(0)
        # --- 训练D ---
        z = torch.randn(bs, nz).to(device)
        fake = G(z).detach()  # detach：不传梯度给G
        loss_D = criterion(D(real), torch.ones(bs,1).to(device)) + \
                 criterion(D(fake), torch.zeros(bs,1).to(device))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # --- 训练G ---
        z = torch.randn(bs, nz).to(device)
        loss_G = criterion(D(G(z)), torch.ones(bs,1).to(device))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    print(f"Epoch {epoch+1}, D_loss: {loss_D.item():.4f}, G_loss: {loss_G.item():.4f}")

Epoch 1, D_loss: 0.8183, G_loss: 1.8906
Epoch 2, D_loss: 1.0290, G_loss: 2.2759
Epoch 3, D_loss: 1.7814, G_loss: 0.4036
Epoch 4, D_loss: 0.9980, G_loss: 1.8451
Epoch 5, D_loss: 0.9312, G_loss: 1.4739
Epoch 6, D_loss: 0.9562, G_loss: 1.5977
Epoch 7, D_loss: 1.1372, G_loss: 1.6573
Epoch 8, D_loss: 1.0367, G_loss: 1.4363
Epoch 9, D_loss: 1.0450, G_loss: 0.9345
Epoch 10, D_loss: 1.3105, G_loss: 0.5161


### 3.10 VAE（变分自编码器）

> **做什么**：编码器学潜在分布(z_mu, z_sigma)，解码器从z重建图像  
> **经典案例**：MNIST变分自编码器

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cpu')

transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()
        # 编码器：图像->(mu, log_var)
        self.encoder = nn.Sequential(nn.Linear(28*28, 256), nn.ReLU(),
                                     nn.Linear(256, 128), nn.ReLU())
        self.fc_mu = nn.Linear(128, latent_dim)      # 均值
        self.fc_logvar = nn.Linear(128, latent_dim)   # 对数方差
        # 解码器：z->重建图像
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                     nn.Linear(128, 256), nn.ReLU(),
                                     nn.Linear(256, 28*28), nn.Sigmoid())

    def reparameterize(self, mu, logvar):
        """重参数化技巧: z = mu + sigma*eps, eps~N(0,1)"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x.view(-1, 28*28))
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)  # 采样
        recon = self.decoder(z)
        return recon, mu, logvar

def vae_loss(recon, x, mu, logvar):
    """重建损失 + KL散度"""
    BCE = nn.functional.binary_cross_entropy(recon, x.view(-1,28*28), reduction='sum')
    KL = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KL

model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    total_loss = 0
    for Xb, _ in loader:
        Xb = Xb.to(device)
        recon, mu, logvar = model(Xb)
        loss = vae_loss(recon, Xb, mu, logvar)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset):.4f}")

Epoch 1, Loss: 170.8919
Epoch 2, Loss: 126.3381
Epoch 3, Loss: 117.4183
Epoch 4, Loss: 113.3709
Epoch 5, Loss: 111.1021
Epoch 6, Loss: 109.5008
Epoch 7, Loss: 108.4226
Epoch 8, Loss: 107.5424
Epoch 9, Loss: 106.8832
Epoch 10, Loss: 106.2691


### 3.11 Diffusion（简化版去噪扩散）

> **做什么**：逐步加噪->学习逐步去噪->从纯噪声生成数据  
> **经典案例**：1D/2D简单演示（不跑Stable Diffusion大模型）

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = torch.device('cpu')
T = 200  # 扩散步数
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T).to(device)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)  # alpha_bar_t

# ====== 1. 前向扩散：逐步加噪 ======
def q_sample(x_0, t, noise=None):
    """给定x_0和时间步t, 采样x_t"""
    if noise is None:
        noise = torch.randn_like(x_0)
    sqrt_alpha = torch.sqrt(alpha_cumprod[t])[:, None]
    sqrt_one_minus_alpha = torch.sqrt(1 - alpha_cumprod[t])[:, None]
    return sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise

# ====== 2. 简单去噪网络 ======
class SimpleDenoiser(nn.Module):
    def __init__(self, dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 64), nn.ReLU(),   # +1: 时间步embedding
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x, t):
        t_norm = (t.float() / T).unsqueeze(1)  # 归一化时间步
        return self.net(torch.cat([x, t_norm], dim=1))

# ====== 3. 训练：学习预测噪声 ======
model = SimpleDenoiser().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 生成2D螺旋数据
theta = torch.linspace(0, 4*np.pi, 500)
x_0 = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1).to(device) * 0.5

for epoch in range(500):
    t = torch.randint(0, T, (x_0.size(0),), device=device)  # 随机时间步
    noise = torch.randn_like(x_0)
    x_t = q_sample(x_0, t, noise)        # 加噪到x_t
    pred_noise = model(x_t, t)            # 预测噪声
    loss = nn.functional.mse_loss(pred_noise, noise)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

# ====== 4. 反向采样：从噪声逐步去噪 ======
with torch.no_grad():
    x = torch.randn(100, 2).to(device)  # 从纯噪声开始
    for t_val in reversed(range(T)):
        t_batch = torch.full((100,), t_val, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha_t = alphas[t_val]
        alpha_bar_t = alpha_cumprod[t_val]
        # DDPM采样公式
        x = (x - (betas[t_val]/torch.sqrt(1-alpha_bar_t)) * pred_noise) / torch.sqrt(alpha_t)
        if t_val > 0:
            x += torch.sqrt(betas[t_val]) * torch.randn_like(x)
print(f"Generated mean: {x.mean(0).cpu().numpy().round(3)}")
print(f"Generated std:  {x.std(0).cpu().numpy().round(3)}")

Epoch 100, Loss: 0.302861
Epoch 200, Loss: 0.286120
Epoch 300, Loss: 0.273978
Epoch 400, Loss: 0.273944
Epoch 500, Loss: 0.284633
Generated mean: [ 0.057 -0.071]
Generated std:  [0.3   0.333]
